# Mellyn Security — Colab A100 Open-Weight Evaluation Suite

Runs all three external security harnesses locally on an **A100 GPU (40GB/80GB VRAM)** with open-weight **`Qwen/Qwen3-32B-AWQ`** (or optional 80GB preset **`Qwen/Qwen3-30B-A3B-Instruct-2507`**) via **vLLM** (`http://127.0.0.1:8000/v1`):
1. **`mcp-fuzzer` (`evals/fuzzer/`)** — Protocol fuzzing & adversarial canary/Unicode/path probes against `mellyn.airlock.proxy`.
2. **`Promptfoo` (`evals/promptfoo/`)** — Side-by-side A/B comparison (**Mellyn OFF `direct` vs. Mellyn ON `airlock`**) across 12 prompt-injection, canary-exfiltration, SSRF, guardrail-tampering, and utility cases.
3. **`AgentDojo` (`evals/agentdojo/`)** — Quantitative Benign Utility vs. Attack Success Rate (ASR) across `workspace`, `banking`, `slack`, and `travel` suites (**Baseline vs. `MellynToolGate`**).
4. **`mellyn-ir`** — Cryptographic hash-chain and incident containment verification over every generated `audit.jsonl` ledger.


In [ ]:
%%bash
set -euo pipefail
nvidia-smi

# The commit (or branch) to evaluate. It must contain evals/, so push it to GitHub first; a full commit SHA
# is best, because a branch can move between runs. The resolved SHA is written to outputs/target_commit.txt.
TARGET_REF="evals/colab-a100-harness"

if [ ! -d "mellyn_security" ] && [ ! -f "pyproject.toml" ]; then
  git clone https://github.com/OramramLLC/mellyn_security.git
  git -C mellyn_security fetch --quiet origin "$TARGET_REF"
  git -C mellyn_security checkout --quiet --detach FETCH_HEAD
fi
if [ -d "mellyn_security" ]; then
  cd mellyn_security
fi
if [ ! -d "evals" ]; then
  echo "evals/ is missing from this checkout; set TARGET_REF to a commit that contains it." >&2
  exit 1
fi
mkdir -p outputs
git rev-parse HEAD > outputs/target_commit.txt 2>/dev/null || echo "not-a-git-checkout" > outputs/target_commit.txt
echo "Evaluating commit: $(cat outputs/target_commit.txt)"

python -m pip install -q --upgrade pip
# agentdojo is pinned because evals/agentdojo is written against the 0.1.35 API.
python -m pip install -q -e . vllm mcp-fuzzer agentdojo==0.1.35
node -v && npm -v


In [ ]:
import http.client
import os
import subprocess
import sys
import time
from pathlib import Path

if Path("mellyn_security").is_dir():
    os.chdir("mellyn_security")

# Set True if nvidia-smi shows an 80GB A100 and you want the faster non-thinking BF16 MoE.
USE_80GB_MOE_PRESET = False
MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507" if USE_80GB_MOE_PRESET else "Qwen/Qwen3-32B-AWQ"
BASE_URL = "http://127.0.0.1:8000/v1"
os.environ["OPENAI_BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["MELLYN_EVAL_MODEL"] = MODEL_NAME

Path("outputs").mkdir(parents=True, exist_ok=True)
vllm_log = open("outputs/vllm_server.log", "w", encoding="utf-8")

vllm_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    MODEL_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
    "--enable-auto-tool-choice",
    "--tool-call-parser",
    "hermes",
    "--max-model-len",
    "16384",
    "--gpu-memory-utilization",
    "0.90",
]
if not USE_80GB_MOE_PRESET:
    vllm_cmd.extend(["--reasoning-parser", "qwen3", "--default-chat-template-kwargs", '{"enable_thinking": false}'])

vllm_proc = subprocess.Popen(  # noqa: S603 - starts local pinned vLLM module with static model flags
    vllm_cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)

for attempt in range(90):
    conn = http.client.HTTPConnection("127.0.0.1", 8000, timeout=2.0)
    try:
        conn.request("GET", "/v1/models")
        resp = conn.getresponse()
        if resp.status == 200:
            print(f"vLLM ready with {MODEL_NAME} after {attempt * 5}s")
            break
    except (OSError, http.client.HTTPException):
        time.sleep(5)
    finally:
        conn.close()
else:
    raise RuntimeError("vLLM failed to become healthy. Inspect outputs/vllm_server.log.")

In [ ]:
%%bash
set -euo pipefail
python -m evals.fuzzer.run_fuzzer --mode smoke --output-dir outputs/fuzzer_smoke
python -m evals.fuzzer.run_fuzzer --mode mcp-fuzzer --runs 25 --output-dir outputs/fuzzer_live


In [ ]:
%%bash
set -euo pipefail
mkdir -p outputs/promptfoo
npx -y promptfoo@latest eval -c evals/promptfoo/promptfooconfig.yaml -o outputs/promptfoo/results.json --no-cache


In [ ]:
%%bash
set -euo pipefail
python -m evals.agentdojo.run_agentdojo --suites workspace banking slack travel --output-dir outputs/agentdojo
cat outputs/agentdojo/agentdojo_scorecard.md


In [ ]:
import subprocess
import sys
from pathlib import Path

ledgers = sorted(Path("outputs").rglob("*audit.jsonl"))
print(f"Verifying {len(ledgers)} audit ledger(s)...")
for ledger in ledgers:
    res = subprocess.run(  # noqa: S603 - invokes pinned local mellyn.ir_lab module over output ledger paths
        [sys.executable, "-m", "mellyn.ir_lab.reconstruct", "--json", str(ledger)],
        capture_output=True,
        text=True,
        check=False,
    )
    print(f"{ledger}: exit={res.returncode}")
    if res.returncode != 0:
        print(res.stdout or res.stderr)